# ARVI-RX — Couche 2 (ML) : Baseline VLM (MedGemma)

Ce notebook fait tourner **`vlm_predict`** (MedGemma-4b-it) sur les cas du projet et produit
le **tableau baseline vs improved** (métriques) pour la section *Résultats* du livrable.

**Avant de lancer :**
1. `Exécution > Modifier le type d'exécution > GPU` (T4 suffit).
2. Avoir un compte Hugging Face et **avoir accepté la licence** de `google/medgemma-4b-it`
   (page du modèle > *Agree and access repository*). Sinon le téléchargement échouera.
3. Créer un **token HF** (Settings > Access Tokens, role *read*).

> ⚠️ Les images du repo sont **synthétiques** (carrés gris). Le pipeline tourne, mais les
> résultats médicaux n'ont de sens que sur de **vraies radios** (voir la dernière section : RSNA).

## 1. Installation des dépendances

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub "pillow>=11.0" pandas
# IMPORTANT : si tu vois une erreur Pillow (ex. "cannot import name '_Ink'"),
# redemarre la session (Execution > Redemarrer la session) puis reprends a la cellule 2.

## 2. Connexion Hugging Face (modèle gated)
Colle ton token quand le champ apparaît.

In [ ]:
from huggingface_hub import login
login()  # colle ton token HF (role: read)

## 3. Récupération du code du projet
On clone le repo sur la branche **`partie-ml`** (qui contient `vlm_predict`).

In [ ]:
import os
REPO_URL = "https://github.com/ThomasSoulliaert/ARVI-RX_DS6B.git"
BRANCH = "partie-ml"
if not os.path.isdir("ARVI-RX_DS6B"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd ARVI-RX_DS6B
!git checkout {BRANCH} && git pull

## 4. Vérification GPU + import du code

In [ ]:
import torch
print("GPU dispo :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

from src.inference import vlm_predict, MODEL_ID
from src.guardrails import apply_safety_guardrails
print("Modele cible :", MODEL_ID)

## 5. Test sur UNE image (sanity check)
Le 1er appel télécharge le modèle (~8 Go, quelques minutes). Ensuite c'est en cache.

In [ ]:
import json
raw = vlm_predict(
    "data/sample_images/CXR_SYN_002_suspected_opacity.png",
    prompt_path="prompts/baseline_prompt.txt",
    mode="baseline",
)
safe = apply_safety_guardrails(raw)  # toujours passer par les garde-fous
print(json.dumps(safe, indent=2, ensure_ascii=False))

## 6. Évaluation complète : baseline vs improved
On boucle sur les cas du CSV, pour les **deux prompts**, avec garde-fous.

Mets `LIMIT = None` pour tout faire, ou un petit nombre pour un essai rapide.

In [ ]:
import csv
import pandas as pd
from src.guardrails import validate_prediction

LIMIT = 12  # -> None pour tout le dataset
CASES_CSV = "data/synthetic_cases.csv"
PROMPTS = {
    "baseline": "prompts/baseline_prompt.txt",
    "improved": "prompts/improved_prompt.txt",
}

with open(CASES_CSV, newline="", encoding="utf-8") as f:
    cases = list(csv.DictReader(f))
if LIMIT:
    cases = cases[:LIMIT]

records = []
for mode, prompt_path in PROMPTS.items():
    for case in cases:
        raw = vlm_predict(case["image_path"], prompt_path=prompt_path, mode=mode)
        pred = apply_safety_guardrails(raw)
        valid, _ = validate_prediction(pred)
        records.append({
            "mode": mode,
            "case_id": case["case_id"],
            "label": case["label"],
            "predicted_class": pred["predicted_class"],
            "confidence": pred["confidence"],
            "json_valid": valid,
            "latency_ms": pred["latency_ms"],
        })
    print(f"[{mode}] termine ({len(cases)} cas)")

df = pd.DataFrame(records)
df

## 7. Tableau de métriques (à mettre dans la section Résultats)
Pour le médical, la **sensibilité** (détecter les opacités) compte plus que l'accuracy brute :
un faux négatif (rater une opacité) est plus grave qu'une fausse alerte.

In [ ]:
POSITIVE = "suspected_opacity"

def metrics_for(sub: pd.DataFrame) -> dict:
    acc = (sub["predicted_class"] == sub["label"]).mean()
    pos = sub[sub["label"] == POSITIVE]
    tp = (pos["predicted_class"] == POSITIVE).sum()
    sensitivity = tp / len(pos) if len(pos) else float("nan")
    neg = sub[sub["label"] != POSITIVE]
    tn = (neg["predicted_class"] != POSITIVE).sum()
    specificity = tn / len(neg) if len(neg) else float("nan")
    return {
        "n": len(sub),
        "accuracy": round(acc, 3),
        "sensitivity_opacity": round(sensitivity, 3),
        "specificity": round(specificity, 3),
        "uncertain_rate": round((sub["predicted_class"] == "uncertain").mean(), 3),
        "json_valid_rate": round(sub["json_valid"].mean(), 3),
        "latency_ms_med": int(sub["latency_ms"].median()),
    }

summary = pd.DataFrame([{"mode": m, **metrics_for(g)} for m, g in df.groupby("mode")])
summary = summary.set_index("mode")
print(summary.to_string())
summary

## 8. Matrice de confusion (par mode)

In [ ]:
for mode, g in df.groupby("mode"):
    print(f"=== {mode} ===")
    cm = pd.crosstab(g["label"], g["predicted_class"], rownames=["vrai"], colnames=["predit"])
    print(cm, "\n")

## 9. Export des résultats (pour le rapport / la couche dataviz)

In [ ]:
df.to_csv("vlm_predictions_detail.csv", index=False)
summary.to_csv("vlm_summary_baseline_vs_improved.csv")
print("Ecrit : vlm_predictions_detail.csv  +  vlm_summary_baseline_vs_improved.csv")
from google.colab import files
files.download("vlm_summary_baseline_vs_improved.csv")

## 10. Tester sur de vraies radios — RSNA Pneumonia (Kaggle)

C'est l'étape qui donne des résultats **défendables** (les images synthétiques ne veulent rien dire).

**Prérequis Kaggle :**
1. Avoir un compte Kaggle et **accepter les règles de la compétition**
   (page *RSNA Pneumonia Detection Challenge* > onglet *Rules* > *I Understand and Accept*), sinon erreur 403.
2. Créer un token API : Kaggle > *Settings* > *API* > *Create New API Token* → tu obtiens un token `KGAT_...`
   (tu le colleras en cellule 10.1, saisie masquée).

On télécharge le dataset, puis on **échantillonne ~30 cas équilibrés** (normal / opacité) et on
convertit les DICOM en PNG. Le mapping : `Target=1` → `suspected_opacity`, `Target=0` → `normal`.

In [ ]:
# 10.1 Authentification Kaggle (nouveau format : token KGAT_...)
!pip install -q kaggle
import os
from getpass import getpass

# Saisie masquee : le token n'est PAS ecrit dans le notebook.
token = getpass("Colle ton token Kaggle (KGAT_...) puis Entree : ").strip()
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(token)
os.chmod("/root/.kaggle/access_token", 0o600)
os.environ["KAGGLE_API_TOKEN"] = token  # certains clients lisent la variable d'env

# Verification : doit lister des competitions sans erreur 401/403.
!kaggle competitions list | head -n 5

In [ ]:
# 10.2 Téléchargement du dataset RSNA (~3.5 Go, quelques minutes)
!kaggle competitions download -c rsna-pneumonia-detection-challenge
!ls -lh rsna-pneumonia-detection-challenge.zip

In [ ]:
# 10.3 Échantillonnage équilibré + conversion DICOM -> PNG + CSV
!pip install -q pydicom

import zipfile, csv, os, random
import numpy as np
import pydicom
from PIL import Image

ZIP = "rsna-pneumonia-detection-challenge.zip"
N_PER_CLASS = 15  # 15 normal + 15 opacite = 30 cas
os.makedirs("data/real_images", exist_ok=True)

zf = zipfile.ZipFile(ZIP)

# 1. Labels : 1 patient = 1 etiquette (Target=1 => opacite). On deduplique.
zf.extract("stage_2_train_labels.csv", ".")
with open("stage_2_train_labels.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
label_by_pid = {}
for r in rows:
    pid = r["patientId"]
    label_by_pid[pid] = max(label_by_pid.get(pid, 0), int(r["Target"]))

opacity = [p for p, t in label_by_pid.items() if t == 1]
normal = [p for p, t in label_by_pid.items() if t == 0]
random.seed(0)
sample = ([(p, "suspected_opacity") for p in random.sample(opacity, N_PER_CLASS)]
          + [(p, "normal") for p in random.sample(normal, N_PER_CLASS)])

# 2. Extraction selective des DICOM choisis + conversion PNG
def dcm_to_png(pid, out_path):
    member = f"stage_2_train_images/{pid}.dcm"
    zf.extract(member, ".")
    ds = pydicom.dcmread(member)
    arr = ds.pixel_array.astype(float)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = arr.max() - arr  # inverser les radios en negatif
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255.0
    Image.fromarray(arr.astype(np.uint8)).convert("RGB").save(out_path)

# 3. Ecriture du CSV au meme format que synthetic_cases.csv
with open("data/real_cases.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["case_id", "image_path", "label"])
    for pid, label in sample:
        out_path = f"data/real_images/{pid}.png"
        dcm_to_png(pid, out_path)
        w.writerow([pid, out_path, label])

print(f"OK : {len(sample)} images -> data/real_images/  |  CSV -> data/real_cases.csv")

### 10.4 Relancer l'évaluation sur les vraies images
Reviens à la **cellule 6**, mets :
```python
CASES_CSV = "data/real_cases.csv"
LIMIT = None
```
puis ré-exécute les cellules **6 → 9**. Le tableau (cellule 7) et la matrice de confusion (cellule 8)
seront alors calculés sur de vraies radios → c'est ça que tu mets dans le livrable.